# logit-thermo 探索：凝固点降下の下見（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 5〜15 分。
- 探索（予想なし）。A2-4（なし・写しの禁止・偽薬）と A2-7（n ＝ 8・6・3・2）のデータから、「邪魔した確率の量」と「融点の下がり方」を並べて `A2_explore/colligative_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：A2-2・A2-4 の分析のコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_explore_colligative.py`（`1201e27cc387e80d…`）は埋め込み。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo 探索 凝固点降下（邪魔の量と融点、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 凝固点降下（邪魔の量と融点、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore colligative', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRCA22': 'fe1f96db1f7e84031e85a70bcc8dcbb95c9fc06b3367657ff4074a9f9d1343f7', 'SRCA24': 'e70bad0db9b7780c2059aae52ad06debe87757bc582791deefb58c5f435f84a6'})
SRCA22, SRCA24 = _s['SRCA22'], _s['SRCA24']
SRCCL = '"""探索：凝固点降下（束一的性質）の下見。2026-10-29 夕方。確かめには使わない（予想なし）。CPU。\nscripts/a2_2_analyze.py（per_doc_feedback・feedback_stats）と scripts/a2_4_analyze.py（stats_fixed）を実行したのと同じ名前空間で実行する。\n\n発想（オーナー 2026-10-29 18:06）：（オーナーの承認・発言。原文は非公開の記録にある）\n問い：融点の下がり方 log(T_0(邪魔) ÷ T_0(なし)) は、邪魔の「種類」によらず「量」だけで決まるか。\n  邪魔の種類：写しの禁止（n ＝ 8・6・4・3・2。n ＝ 4 は A2-4 の block）と、A2-4 の偽薬（写しかけでない歩の 1 位を同じ割合で禁止）。\n  邪魔の量：禁止したトークンの、その温度の分布での確率の合計（g_ban_mass）の、全歩・全サンプル・全記事の平均。\n            温度 0.8・0.85・0.9（T_0 のまわり）の平均を主に使い、温度ごとの値も残す。\nT_0 は A2-7 と同じ（温度 8 つ、ε は「なし」で出したもの）。交点が出ないときは記録だけ。\n"""\nimport math\nimport numpy as np\n\nCL_MODELS = [\'pythia-1b\', \'olmo2-1b\', \'qwen2.5-1.5b\']\nCL_TEMPS = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05]\nCL_MASS_T = [0.8, 0.85, 0.9]\nCL_DOSES = [(\'n8\', \'A2_7\', \'@n8\'), (\'n6\', \'A2_7\', \'@n6\'), (\'n4\', \'A2_4\', \'@block\'), (\'n3\', \'A2_7\', \'@n3\'), (\'n2\', \'A2_7\', \'@n2\'),\n            (\'placebo\', \'A2_4\', \'@placebo\')]\n\n\ndef run_colligative(base):\n    res = {}\n    for m in CL_MODELS:\n        Zn = np.load(f\'{base}/A2_4/{m}@none.npz\')\n        t4 = [float(t) for t in Zn[\'temps\']]\n        kk = [t4.index(t) for t in CL_TEMPS]\n        tf = Zn[\'tf\'][0][:, kk]\n        fd0 = per_doc_feedback(tf, Zn[\'g_ell\'][kk])\n        eps = feedback_stats(fd0, CL_TEMPS, np.random.default_rng(0))[\'eps\']\n        s0 = stats_fixed(fd0, CL_TEMPS, eps)\n        row = {\'none\': dict(T0=s0[\'T0\'], cls=s0[\'cls\'], mass=0.0)}\n        for dose, folder, suf in CL_DOSES:\n            Z = np.load(f\'{base}/{folder}/{m}{suf}.npz\')\n            tz = [float(t) for t in Z[\'temps\']]\n            kz = [tz.index(t) for t in CL_TEMPS]\n            g = Z[\'g_ell\'][kz]\n            s = stats_fixed(per_doc_feedback(tf, g), CL_TEMPS, eps)\n            bm = Z[\'g_ban_mass\'][kz].astype(np.float64)                 # (8, S, nD, L)\n            mass_T = bm.mean(axis=(1, 2, 3))\n            mass = float(np.mean([mass_T[CL_TEMPS.index(t)] for t in CL_MASS_T]))\n            top1 = Z[\'g_banned_top1\'][kz].astype(np.float64).mean(axis=(1, 2, 3))\n            row[dose] = dict(T0=s[\'T0\'], cls=s[\'cls\'], mass=mass, mass_by_T=mass_T.tolist(),\n                             top1_rate_by_T=top1.tolist(),\n                             dlogT0=(math.log(s[\'T0\'] / s0[\'T0\']) if np.isfinite(s[\'T0\']) and np.isfinite(s0[\'T0\']) else float(\'nan\')))\n        # 写しの禁止の点（交点が出たものだけ）で、原点を通る直線 dlogT0 ＝ −K × 量 を当てはめ、偽薬がそれに乗るかを見る\n        pts = [(row[d][\'mass\'], row[d][\'dlogT0\']) for d, _, _ in CL_DOSES if d != \'placebo\' and np.isfinite(row[d][\'dlogT0\'])]\n        if len(pts) >= 2:\n            x = np.array([p[0] for p in pts]); y = np.array([p[1] for p in pts])\n            K = float(-(x @ y) / (x @ x))\n            pl = row[\'placebo\']\n            pred = -K * pl[\'mass\']\n            row[\'fit\'] = dict(K=K, n_points=len(pts), placebo_pred=pred, placebo_obs=pl[\'dlogT0\'],\n                              resid_rms=float(np.sqrt(np.mean((y + K * x) ** 2))))\n        res[m] = row\n    return res\n\n\ndef self_test_colligative():\n    x = np.array([0.1, 0.2, 0.4]); y = -0.5 * x\n    K = float(-(x @ y) / (x @ x))\n    assert abs(K - 0.5) < 1e-12\n    return \'OK\'\n'
assert hashlib.sha256(SRCCL.encode()).hexdigest() == '1201e27cc387e80d9ed4dc7577e6ad177fef5ca010a2813bde26df6472ee55fb'
exec(SRCA22)
exec(SRCA24)
exec(SRCCL)
print('自己テスト：', self_test_colligative())


In [ ]:
# セル3：数えて保存（結果は表示しない）
import json
try:
    r = run_colligative('/content/drive/MyDrive/logit-thermo')
    r['a2_explore_colligative_sha256'] = '1201e27cc387e80d9ed4dc7577e6ad177fef5ca010a2813bde26df6472ee55fb'
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo' + '/A2_explore/colligative_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 凝固点降下で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：邪魔の量と融点を A2_explore/colligative_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
